# Milestone 10 — Matched hard-label and distilled BGE students

Select a GPU runtime, then run the cells in order. Upload the supplied **finevid_milestone10_source.zip** when prompted. This bundle contains the reviewed implementation, tests, and training/development ranking pools; it requires no GitHub token.

Both treatments start from the pinned pretrained BGE model, use the same seed-42 training rows and three-epoch budget, and use student temperature 0.05. Distillation adds KL(teacher || student) with teacher temperature 0.3 and a 0.1 hard-label anchor. There is no temperature-squared multiplier.

The original `hard_label_student` folder is preserved. New checkpoints go to `hard_label_student_tau005` and `distilled_student`. Rerun this notebook with the same bundle to resume. A CPU smoke test does not satisfy the full GPU experiment.


In [ ]:
from google.colab import drive, files
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import hashlib
import json
import subprocess
import sys

drive.mount('/content/drive')
uploaded = files.upload()  # Select finevid_milestone10_source.zip.
if len(uploaded) != 1:
    raise ValueError('Upload exactly one source bundle.')
bundle_name, bundle_bytes = next(iter(uploaded.items()))
bundle_hash = hashlib.sha256(bundle_bytes).hexdigest()
PROJECT_DIR = Path('/content') / ('finevid-distill-m10-' + bundle_hash[:12])
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
with ZipFile(BytesIO(bundle_bytes)) as archive:
    manifest = json.loads(archive.read('bundle_manifest.json'))
    expected = set(manifest['files']) | {'bundle_manifest.json'}
    if len(archive.namelist()) != len(expected) or set(archive.namelist()) != expected:
        raise ValueError('Bundle file list does not match its manifest.')
    for name, expected_hash in manifest['files'].items():
        destination = (PROJECT_DIR / name).resolve()
        if not destination.is_relative_to(PROJECT_DIR.resolve()):
            raise ValueError('Invalid archive path.')
        if hashlib.sha256(archive.read(name)).hexdigest() != expected_hash:
            raise ValueError('Bundle integrity check failed: ' + name)
    archive.extractall(PROJECT_DIR)
del uploaded, bundle_bytes
ARTIFACT_ROOT = Path('/content/drive/MyDrive/FinEvid-Distill')
TEACHER_DIR = ARTIFACT_ROOT / 'teacher_scores'
PROCESSED_DIR = ARTIFACT_ROOT / 'processed_data'
HARD_DIR = ARTIFACT_ROOT / 'checkpoints' / 'hard_label_student_tau005'
DISTILLED_DIR = ARTIFACT_ROOT / 'checkpoints' / 'distilled_student'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
assert (TEACHER_DIR / 'teacher_train_scores.jsonl').is_file(), 'The completed teacher cache must be on Drive.'
print('Verified code bundle:', bundle_hash)
print('Code directory:', PROJECT_DIR)


## Install and check the GPU

Colab's existing CUDA-enabled PyTorch is retained. Training and tests run in fresh Python processes, avoiding stale notebook imports.


In [ ]:
%pip install -q -r {PROJECT_DIR / 'requirements-colab.txt'}
%pip install -q -e {PROJECT_DIR} --no-deps


In [ ]:
def run_project(*arguments):
    return subprocess.run([sys.executable, *arguments], cwd=PROJECT_DIR, check=True)

run_project('-c', "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name(0)); print('PyTorch:', torch.__version__)")
run_project('-m', 'pytest', '-q')


## Validate the shared rows

Rebuild both treatments' identical candidate rows from the existing teacher cache. Teacher scores remain raw in storage; temperature is applied only inside the distilled loss.


In [ ]:
TRAIN_ROWS = PROCESSED_DIR / 'train_rows.jsonl'
run_project(
    'src/data/build_training_rows.py',
    '--teacher-cache', str(TEACHER_DIR / 'teacher_train_scores.jsonl'),
    '--output', str(TRAIN_ROWS),
)

def run_treatment(script, checkpoint_dir):
    arguments = [
        script,
        '--config', str(PROJECT_DIR / 'configs/beginner.yaml'),
        '--train-rows', str(TRAIN_ROWS),
        '--dev-data', str(PROJECT_DIR / 'data/processed/dev.jsonl'),
        '--output-dir', str(checkpoint_dir),
        '--device', 'cuda',
    ]
    if (checkpoint_dir / 'run_config.json').exists():
        arguments.append('--resume')
    run_project(*arguments)


## Train the matched hard-label control

This is a fresh run from pretrained weights at student temperature 0.05. Each trainer evaluates the pretrained model before its first optimizer step and reevaluates the best saved checkpoint after training.


In [ ]:
run_treatment('src/training/train_hard_labels.py', HARD_DIR)
print((HARD_DIR / 'best_reload_verification.json').read_text())


## Train the distilled student

The student starts independently from the same pretrained weights, not from the trained hard-label checkpoint. The log records KL, hard-label, total loss, teacher entropy, and development metrics.


In [ ]:
run_treatment('src/training/train_distilled.py', DISTILLED_DIR)
print((DISTILLED_DIR / 'best_reload_verification.json').read_text())


## Verify the comparison

The comparison fails if the two runs differ in data hashes, model revision, code, training budget, student temperature, or initial development metrics. It also checks that both completed their budgets and reloaded their best checkpoints. The final held-out evaluation remains a later milestone.


In [ ]:
COMPARISON_PATH = ARTIFACT_ROOT / 'dev_student_comparison.json'
run_project(
    'src/evaluation/compare_students.py',
    '--hard-dir', str(HARD_DIR),
    '--distilled-dir', str(DISTILLED_DIR),
    '--output', str(COMPARISON_PATH),
)
comparison = json.loads(COMPARISON_PATH.read_text())
print(json.dumps(comparison, indent=2))


## Download the review records

Download this small ZIP and attach it for review. The trained model checkpoints stay in Google Drive.


In [ ]:
from zipfile import ZIP_DEFLATED

review_zip = Path('/content/milestone10_run_json.zip')
record_names = [
    'run_config.json', 'initial_development.json', 'training_history.json',
    'latest.json', 'best.json', 'best_reload_verification.json',
]
with ZipFile(review_zip, 'w', ZIP_DEFLATED) as archive:
    for directory in (HARD_DIR, DISTILLED_DIR):
        for name in record_names:
            archive.write(directory / name, arcname=directory.name + '/' + name)
    archive.write(COMPARISON_PATH, arcname=COMPARISON_PATH.name)
    archive.writestr('source_bundle_sha256.txt', bundle_hash)
files.download(str(review_zip))
